In [1]:
!pip install gradio sentence-transformers numpy torch -q


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import re
from typing import List, Dict, Tuple

import gradio as gr
import numpy as np
from sentence_transformers import SentenceTransformer, util

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

SKILL_BANK = {
    "programming": [
        "python", "java", "c++", "c", "javascript", "typescript", "sql", "html", "css"
    ],
    "frameworks_tools": [
        "react", "node", "flask", "fastapi", "docker", "git", "github", "linux", "aws", "azure", "api", "rest"
    ],
    "ml_data": [
        "machine learning", "deep learning", "nlp", "computer vision", "data analysis",
        "data science", "pandas", "numpy", "matplotlib", "tensorflow", "pytorch", "scikit-learn"
    ],
    "soft_skills": [
        "leadership", "teamwork", "problem solving", "time management", "adaptability", "communication"
    ],
    "education_experience": [
        "bachelor", "degree", "computer science", "internship", "project", "research", "experience"
    ]
}

EXAMPLE_RESUME = """Computer Science student with experience in Python, Java, SQL, Git, and machine learning.
Built a full-stack multiplayer Tic-Tac-Toe application using client-server communication and multithreading.
Created data analysis projects with pandas, NumPy, and matplotlib.
Strong teamwork, communication, and problem-solving skills developed through academic group projects.
Completed software development coursework and worked on API-based applications."""

EXAMPLE_JOB = """We are seeking a junior software developer with experience in Python, SQL, Git, and API development.
Candidates should have strong problem solving, teamwork, and communication skills.
Experience with machine learning or data analysis is considered an asset.
Applicants should be pursuing a degree in Computer Science or a related field and have project experience."""

def normalize_text(text: str) -> str:
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def split_into_lines(text: str) -> List[str]:
    raw = re.split(r"\n+|(?<=[.!?])\s+", text)
    lines = []
    for item in raw:
        item = item.strip(" -•\t")
        if len(item.split()) >= 3:
            lines.append(item)
    return lines

def chunk_text(text: str, max_words: int = 40) -> List[str]:
    lines = split_into_lines(normalize_text(text))
    if not lines:
        return []

    chunks = []
    current_chunk = []
    word_count = 0

    for line in lines:
        line_words = line.split()
        if current_chunk and word_count + len(line_words) > max_words:
            chunks.append(" ".join(current_chunk))
            current_chunk = [line]
            word_count = len(line_words)
        else:
            current_chunk.append(line)
            word_count += len(line_words)

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

def extract_keywords_from_bank(text: str) -> Dict[str, List[str]]:
    text_lower = text.lower()
    found = {}

    for category, keywords in SKILL_BANK.items():
        found_items = []
        for keyword in keywords:
            pattern = r"(?<!\w)" + re.escape(keyword) + r"(?!\w)"
            if re.search(pattern, text_lower):
                found_items.append(keyword)
        found[category] = sorted(found_items)

    return found

def flatten_keyword_dict(data: Dict[str, List[str]]) -> List[str]:
    items = []
    for values in data.values():
        items.extend(values)
    return sorted(set(items))

def score_to_label(score: float) -> str:
    if score >= 85:
        return "Excellent Match"
    elif score >= 70:
        return "Strong Match"
    elif score >= 55:
        return "Moderate Match"
    elif score >= 40:
        return "Weak Match"
    return "Low Match"

def format_bullets(items: List[str], empty_message: str) -> str:
    if not items:
        return empty_message
    return "\n".join(f"- {item}" for item in items)

def compute_semantic_score(resume_chunks: List[str], job_chunks: List[str]) -> Tuple[float, List[Tuple[float, str, str]]]:
    resume_embeddings = model.encode(
        resume_chunks,
        convert_to_tensor=True,
        normalize_embeddings=True
    )
    job_embeddings = model.encode(
        job_chunks,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    sim_matrix = util.cos_sim(job_embeddings, resume_embeddings).cpu().numpy()
    best_resume_indices = sim_matrix.argmax(axis=1)
    best_scores = sim_matrix.max(axis=1)

    top_matches = []
    for i, score in enumerate(best_scores):
        top_matches.append(
            (float(score), job_chunks[i], resume_chunks[int(best_resume_indices[i])])
        )

    top_matches.sort(reverse=True, key=lambda x: x[0])
    semantic_score = float(np.mean(best_scores) * 100)

    return semantic_score, top_matches[:5]

def compute_keyword_score(job_keywords: List[str], resume_keywords: List[str]) -> Tuple[float, List[str], List[str]]:
    if not job_keywords:
        return 100.0, [], []

    matched = sorted(set(job_keywords) & set(resume_keywords))
    missing = sorted(set(job_keywords) - set(resume_keywords))

    keyword_score = (len(matched) / len(set(job_keywords))) * 100
    return keyword_score, matched, missing

def compute_section_score(job_text: str, resume_text: str) -> Tuple[float, List[str], List[str]]:
    priority_terms = [
        "experience", "project", "projects", "education", "degree", "internship",
        "teamwork", "communication", "leadership", "machine learning", "python", "sql"
    ]

    job_lower = job_text.lower()
    resume_lower = resume_text.lower()

    job_present = [term for term in priority_terms if term in job_lower]
    if not job_present:
        return 100.0, [], []

    matched = [term for term in job_present if term in resume_lower]
    missing = [term for term in job_present if term not in resume_lower]

    section_score = (len(matched) / len(job_present)) * 100
    return section_score, matched, missing

def generate_summary(overall: float, semantic: float, keyword: float, section: float,
                     matched_keywords: List[str], missing_keywords: List[str]) -> str:
    strengths = []
    weaknesses = []

    if semantic >= 75:
        strengths.append("The resume aligns well with the meaning of the job description.")
    elif semantic < 55:
        weaknesses.append("The resume content is not strongly aligned with the main job requirements.")

    if keyword >= 70:
        strengths.append("Many of the important job keywords are present in the resume.")
    elif keyword < 50:
        weaknesses.append("Several important job-specific keywords are missing.")

    if section >= 70:
        strengths.append("The resume includes many of the expected experience or education-related terms.")
    elif section < 50:
        weaknesses.append("The resume appears to be missing important experience or education indicators.")

    if not strengths:
        strengths.append("The resume has some relevant overlap with the posting.")
    if not weaknesses:
        weaknesses.append("There are still opportunities to better tailor the resume to the job description.")

    summary = f"""# Final Analysis
**Overall Match:** {overall:.1f}%
**Rating:** {score_to_label(overall)}

## Strengths
{format_bullets(strengths, 'No major strengths detected.')}

## Weaknesses
{format_bullets(weaknesses, 'No major weaknesses detected.')}

## Missing Keywords
{format_bullets(missing_keywords[:10], 'No major missing keywords detected from the current keyword bank.')}
"""
    return summary

def compare_resume_job(resume_text: str, job_text: str):
    resume_text = normalize_text(resume_text)
    job_text = normalize_text(job_text)

    if not resume_text or not job_text:
        return (
            "Please paste both a resume and a job description.",
            "",
            "",
            "",
            ""
        )

    resume_chunks = chunk_text(resume_text)
    job_chunks = chunk_text(job_text)

    if not resume_chunks or not job_chunks:
        return (
            "Please provide more detailed text in both boxes.",
            "",
            "",
            "",
            ""
        )

    semantic_score, top_matches = compute_semantic_score(resume_chunks, job_chunks)

    job_keywords_dict = extract_keywords_from_bank(job_text)
    resume_keywords_dict = extract_keywords_from_bank(resume_text)

    job_keywords = flatten_keyword_dict(job_keywords_dict)
    resume_keywords = flatten_keyword_dict(resume_keywords_dict)

    keyword_score, matched_keywords, missing_keywords = compute_keyword_score(job_keywords, resume_keywords)
    section_score, matched_sections, missing_sections = compute_section_score(job_text, resume_text)

    overall_score = (
        0.55 * semantic_score +
        0.30 * keyword_score +
        0.15 * section_score
    )

    overall_score = max(0.0, min(100.0, overall_score))

    score_report = f"""# Score Breakdown
**Overall Match Score:** {overall_score:.1f}%
**Rating:** {score_to_label(overall_score)}

- **Semantic Similarity Score:** {semantic_score:.1f}%
- **Keyword Coverage Score:** {keyword_score:.1f}%
- **Experience / Education Score:** {section_score:.1f}%

**Scoring Weights**
- Semantic similarity: 55%
- Keyword coverage: 30%
- Experience/education indicators: 15%
"""

    keyword_report = f"""# Keyword Analysis
## Matched Keywords
{format_bullets(matched_keywords, 'No strong keyword overlap found.')}

## Missing Keywords
{format_bullets(missing_keywords, 'No obvious missing keywords found.')}

## Matched Experience / Education Terms
{format_bullets(matched_sections, 'No matched experience or education terms found.')}

## Missing Experience / Education Terms
{format_bullets(missing_sections, 'No missing experience or education terms found.')}
"""

    alignments = ["# Top Resume-to-Job Alignments"]
    for index, (score, job_chunk, resume_chunk) in enumerate(top_matches, start=1):
        alignments.append(
            f"**Alignment {index} — {score * 100:.1f}%**\n"
            f"**Job Text:** {job_chunk}\n\n"
            f"**Best Resume Evidence:** {resume_chunk}"
        )
    alignment_report = "\n\n".join(alignments)

    category_lines = ["# Category-Level Skill Coverage"]
    for category in SKILL_BANK:
        job_cat = job_keywords_dict.get(category, [])
        resume_cat = resume_keywords_dict.get(category, [])
        matched_cat = sorted(set(job_cat) & set(resume_cat))
        missing_cat = sorted(set(job_cat) - set(resume_cat))

        category_lines.append(
            f"## {category.replace('_', ' ').title()}\n"
            f"**Job requires:** {', '.join(job_cat) if job_cat else 'None detected'}\n\n"
            f"**Matched:** {', '.join(matched_cat) if matched_cat else 'None'}\n\n"
            f"**Missing:** {', '.join(missing_cat) if missing_cat else 'None'}"
        )
    category_report = "\n\n".join(category_lines)

    summary_report = generate_summary(
        overall_score,
        semantic_score,
        keyword_score,
        section_score,
        matched_keywords,
        missing_keywords
    )

    return score_report, keyword_report, alignment_report, category_report, summary_report

def load_example():
    return EXAMPLE_RESUME, EXAMPLE_JOB

with gr.Blocks(title="AI Resume Matcher") as demo:
    gr.Markdown("# AI Resume and Job Description Matching System")
    gr.Markdown(
        "Paste a resume and a job description to analyze semantic similarity, "
        "keyword overlap, and experience/education alignment."
    )

    with gr.Row():
        resume_input = gr.Textbox(
            label="Resume Text",
            lines=18,
            placeholder="Paste the resume here..."
        )
        job_input = gr.Textbox(
            label="Job Description Text",
            lines=18,
            placeholder="Paste the job description here..."
        )

    with gr.Row():
        analyze_button = gr.Button("Analyze Resume Match", variant="primary")
        example_button = gr.Button("Load Example")
        clear_button = gr.Button("Clear")

    with gr.Tabs():
        with gr.Tab("Score Breakdown"):
            score_output = gr.Markdown()

        with gr.Tab("Keyword Analysis"):
            keyword_output = gr.Markdown()

        with gr.Tab("Top Alignments"):
            alignment_output = gr.Markdown()

        with gr.Tab("Category Coverage"):
            category_output = gr.Markdown()

        with gr.Tab("Final Summary"):
            summary_output = gr.Markdown()

    analyze_button.click(
        fn=compare_resume_job,
        inputs=[resume_input, job_input],
        outputs=[
            score_output,
            keyword_output,
            alignment_output,
            category_output,
            summary_output
        ]
    )

    example_button.click(
        fn=load_example,
        inputs=[],
        outputs=[resume_input, job_input]
    )

    clear_button.click(
        fn=lambda: ("", "", "", "", "", "", ""),
        inputs=[],
        outputs=[
            resume_input,
            job_input,
            score_output,
            keyword_output,
            alignment_output,
            category_output,
            summary_output
        ]
    )

if __name__ == "__main__":
    demo.launch(share=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://087f150ddb2c997d77.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
